# BI-0 — Contrato de KPIs (SQL-first)

**Archivo:** `TFM_Fase5_BI_01_Contrato_KPIs_SQL.ipynb`  
**Generado:** 2026-01-23 02:55:07

## Propósito (académico y operativo)

Este notebook define y valida, **mediante SQL ejecutable contra el DW PostgreSQL**, una **lista cerrada** de KPIs (BI-0).  
El resultado es una **tabla oráculo** (`outputs/bi0_oraculo_kpis*.csv`) para reconciliar posteriormente con Power BI (**sin lógica oculta**).

### Reglas
- Fuente única: **DW PostgreSQL**
- KPIs deben ser **reproducibles en SQL**
- NO-OP y EXCEPCIÓN se registran explícitamente en la salida


## 1) Imports y configuración

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


## 2) Parámetros (periodos y estados especiales)

> Ajusta la lista `PERIODOS` si tu alcance cambia, pero mantén la lógica de estados (OK / NO-OP / EXCEPCIÓN).

In [2]:
PERIODOS = [
    '2024-09','2024-10','2024-11','2024-12',
    '2025-01','2025-02','2025-03','2025-04','2025-05',
    '2025-06','2025-07','2025-08','2025-09','2025-10'
]

# Estados especiales (documentados en TFM)
PERIODOS_EXCEPCION = {'2025-01': 'LIC_SCHEMA_ANOM'}   # EXCEPCIÓN
PERIODOS_NOOP = {'2025-03': 'OC_SIZE_ANOM'}           # NO-OP

def estado_periodo(p: str) -> tuple[str, str]:
    if p in PERIODOS_EXCEPCION:
        return 'EXCEPCION', PERIODOS_EXCEPCION[p]
    if p in PERIODOS_NOOP:
        return 'NO-OP', PERIODOS_NOOP[p]
    return 'OK', ''


## 3) Conexión PostgreSQL (DW)

### Variables esperadas (recomendado exportarlas en terminal)
```bash
export PG_HOST=localhost
export PG_PORT=5433
export PG_DB=chilecompra
export PG_USER=chile_user
export PG_PASS='TU_PASSWORD'
```

> **No hardcodear passwords en el notebook.**

In [3]:
from sqlalchemy import create_engine, text

PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = os.getenv('PG_PORT', '5433')
PG_DB   = os.getenv('PG_DB',   'chilecompra')
PG_USER = os.getenv('PG_USER', 'chile_user')
PG_PASS = os.getenv('PG_PASS', 'CHANGE_ME')

if not PG_PASS:
    raise RuntimeError('Falta PG_PASS. Exporta PG_PASS en terminal antes de ejecutar este notebook.')

conn_str = f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}'
engine = create_engine(conn_str)

def read_sql_df(sql: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as con:
        return pd.read_sql(text(sql), con, params=params or {})


## 4) Sanity checks (conectividad + tablas DW mínimas)

In [4]:
display(read_sql_df('SELECT 1 AS ok;'))

sql_tables = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema='dw'
ORDER BY table_name;
"""
df_tables = read_sql_df(sql_tables)
display(df_tables)

# Control de cargas (si existe)
try:
    df_ctl = read_sql_df('SELECT * FROM dw.etl_control_cargas ORDER BY periodo, entidad LIMIT 50;')
    display(df_ctl)
except Exception as e:
    print('Aviso: no se pudo leer dw.etl_control_cargas (puede ser por permisos o inexistencia).', e)


,ok
0,1


,table_schema,table_name
0,dw,dim_fecha
1,dw,dim_organismo
2,dw,dim_producto_onu
3,dw,dim_proveedor
4,dw,etl_control_cargas
5,dw,etl_decisiones_periodo
6,dw,fact_licitaciones
7,dw,fact_ordenes_compra


,entidad,periodo,estado,archivo,filas_stg,filas_dw,ts_inicio,ts_fin,error_msg
0,LIC,2024-09,OK,LIC_2024-09.csv,178091,147755,2026-01-16 13:41:27.271544+00:00,2026-01-16 13:41:27.271544+00:00,None
1,OC,2024-09,OK,OC_2024-09.csv,396516,324419,2026-01-16 13:41:27.271544+00:00,2026-01-16 13:41:27.271544+00:00,None
2,LIC,2024-10,OK,LIC_2024-10.csv,219827,199155,2026-01-16 15:10:54.958475+00:00,2026-01-16 15:22:52.459721+00:00,None
3,OC,2024-10,OK,OC_2024-10.csv,508095,477995,2026-01-16 15:10:54.958475+00:00,2026-01-16 17:07:11.156930+00:00,None
4,LIC,2024-11,OK,LIC_2024-11.csv,530356,475782,2026-01-17 01:55:24.006315+00:00,2026-01-17 15:37:51.834740+00:00,None
5,OC,2024-11,OK,OC_2024-11.csv,470710,429335,2026-01-17 01:55:24.012577+00:00,2026-01-17 15:37:51.841797+00:00,None
6,LIC,2024-12,OK,LIC_2024-12.csv,359662,312983,2026-01-17 17:20:17.988311+00:00,2026-01-17 17:20:17.988311+00:00,None
7,OC,2024-12,OK,OC_2024-12.csv,411596,373131,2026-01-17 17:20:17.995190+00:00,2026-01-17 17:20:17.995190+00:00,None
8,LIC,2025-02,OK,LIC_2025-02.csv,598982,452154,2026-01-20 16:47:32.482900+00:00,2026-01-20 16:47:32.482900+00:00,None
9,OC,2025-02,OK,OC_2025-02.csv,354933,316147,2026-01-20 16:47:32.491460+00:00,2026-01-20 16:47:32.491460+00:00,None


## 5) Inspección de columnas reales (evita suposiciones)

Este bloque detecta columnas en `dw.fact_licitaciones` y `dw.fact_ordenes_compra` y selecciona automáticamente las requeridas.

Si tu DW usa nombres distintos, ajusta **solo** las listas de candidatos.

In [5]:
# 5 - INSPECCIÓN DE COLUMNAS REALES (DW) + MAPEO DEFENSIVO (SIN RECLAMOS)
# Nota: el KPI de reclamos (L4) se DESCARTA porque el campo no existe en dw.fact_licitaciones (verificado por SQL).

def cols(schema: str, table: str) -> pd.DataFrame:
    sql = '''
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema=:schema AND table_name=:table
    ORDER BY ordinal_position;
    '''
    return read_sql_df(sql, {"schema": schema, "table": table})

df_cols_lic = cols("dw", "fact_licitaciones")
df_cols_oc  = cols("dw", "fact_ordenes_compra")

display(df_cols_lic)
display(df_cols_oc)

def pick_col(df_cols: pd.DataFrame, candidates: list[str], label: str) -> str:
    present = set(df_cols["column_name"].tolist())
    for c in candidates:
        if c in present:
            return c
    raise ValueError(
        f"No encontré columna para {label}. "
        f"Candidatas={candidates}. "
        f"Presentes (muestra)={sorted(list(present))[:60]}"
    )

# Ajusta candidatos SOLO si tu DW usa nombres distintos
LIC_BK_COL  = pick_col(df_cols_lic, ["licitacion_bk","id_licitacion_bk","codigo_licitacion","licitacion_id"], "LIC_BK")
LIC_M_ADJ   = pick_col(df_cols_lic, ["monto_adjudicado","monto_adjudicado_clp","monto_adj"], "LIC_MONTO_ADJ")
LIC_PROV_SK = pick_col(df_cols_lic, ["proveedor_sk","sk_proveedor"], "LIC_PROV_SK")
LIC_PERIODO = pick_col(df_cols_lic, ["periodo","periodo_analitico"], "LIC_PERIODO")

OC_BK_COL  = pick_col(df_cols_oc, ["orden_compra_bk","oc_bk","id_oc_bk","ordencompra_bk"], "OC_BK")
OC_MONTO   = pick_col(df_cols_oc, ["monto_total","monto_total_clp","montototaloc","monto_oc_total"], "OC_MONTO")
OC_PROV_SK = pick_col(df_cols_oc, ["proveedor_sk","sk_proveedor"], "OC_PROV_SK")
OC_PERIODO = pick_col(df_cols_oc, ["periodo","periodo_analitico"], "OC_PERIODO")

print("MAPEO OK (sin reclamos):")
print(" LIC_BK_COL :", LIC_BK_COL)
print(" LIC_M_ADJ  :", LIC_M_ADJ)
print(" LIC_PROV   :", LIC_PROV_SK)
print(" LIC_PER    :", LIC_PERIODO)
print(" OC_BK_COL  :", OC_BK_COL)
print(" OC_MONTO   :", OC_MONTO)
print(" OC_PROV    :", OC_PROV_SK)
print(" OC_PER     :", OC_PERIODO)


,column_name,data_type
0,licitacion_sk,bigint
1,licitacion_bk,text
2,fecha_publicacion_sk,integer
3,fecha_cierre_sk,integer
4,organismo_sk,integer
5,proveedor_sk,integer
6,producto_onu_sk,integer
7,monto_estimado,numeric
8,monto_adjudicado,numeric
9,cantidad_oferentes,integer


,column_name,data_type
0,orden_compra_sk,bigint
1,orden_compra_bk,text
2,fecha_creacion_sk,integer
3,fecha_aceptacion_sk,integer
4,organismo_sk,integer
5,proveedor_sk,integer
6,producto_onu_sk,integer
7,monto_total,numeric
8,cantidad_total,numeric
9,cantidad_items,integer


MAPEO OK (sin reclamos):
 LIC_BK_COL : licitacion_bk
 LIC_M_ADJ  : monto_adjudicado
 LIC_PROV   : proveedor_sk
 LIC_PER    : periodo
 OC_BK_COL  : orden_compra_bk
 OC_MONTO   : monto_total
 OC_PROV    : proveedor_sk
 OC_PER     : periodo


## 6) Lista cerrada de KPIs (BI-0)

> **Total: 9 KPIs** (LIC: 3, OC: 3, Cruzados: 3).  
> Los KPIs cruzados son **descriptivos** (no exigen cuadratura mensual).

In [6]:
KPI_CATALOGO = pd.DataFrame([
    {'kpi_id':'L1','dominio':'LIC','nombre':'Monto adjudicado LIC','unidad':'CLP','nota':'SUM(monto_adjudicado)'},
    {'kpi_id':'L2','dominio':'LIC','nombre':'Número de licitaciones','unidad':'conteo','nota':'COUNT DISTINCT licitacion_bk'},
    {'kpi_id':'L3','dominio':'LIC','nombre':'Proveedores adjudicados','unidad':'conteo','nota':'COUNT DISTINCT proveedor_sk'},
    {'kpi_id':'O1','dominio':'OC','nombre':'Monto total OC','unidad':'CLP','nota':'SUM(monto_total)'},
    {'kpi_id':'O2','dominio':'OC','nombre':'Número de OC','unidad':'conteo','nota':'COUNT DISTINCT orden_compra_bk'},
    {'kpi_id':'O3','dominio':'OC','nombre':'Concentración top proveedor (top1/total)','unidad':'ratio','nota':'MAX(monto_prov)/SUM(monto)'},
    {'kpi_id':'C1_LIC','dominio':'CROSS','nombre':'Monto LIC (comparativa)','unidad':'CLP','nota':'SUM(LIC monto_adjudicado)'},
    {'kpi_id':'C1_OC','dominio':'CROSS','nombre':'Monto OC (comparativa)','unidad':'CLP','nota':'SUM(OC monto_total)'},
    {'kpi_id':'C2','dominio':'CROSS','nombre':'Ratio ejecución (OC/LIC)','unidad':'ratio','nota':'SUM(OC)/SUM(LIC)'},
])

display(KPI_CATALOGO)

,kpi_id,dominio,nombre,unidad,nota
0,L1,LIC,Monto adjudicado LIC,CLP,SUM(monto_adjudicado)
1,L2,LIC,Número de licitaciones,conteo,COUNT DISTINCT licitacion_bk
2,L3,LIC,Proveedores adjudicados,conteo,COUNT DISTINCT proveedor_sk
3,O1,OC,Monto total OC,CLP,SUM(monto_total)
4,O2,OC,Número de OC,conteo,COUNT DISTINCT orden_compra_bk
5,O3,OC,Concentración top proveedor (top1/total),ratio,MAX(monto_prov)/SUM(monto)
6,C1_LIC,CROSS,Monto LIC (comparativa),CLP,SUM(LIC monto_adjudicado)
7,C1_OC,CROSS,Monto OC (comparativa),CLP,SUM(OC monto_total)
8,C2,CROSS,Ratio ejecución (OC/LIC),ratio,SUM(OC)/SUM(LIC)


## 7) SQL por KPI (plantillas parametrizadas)

Las consultas usan el mapeo de columnas detectado en el bloque 5.

> Regla: ningún KPI entra a BI si no existe aquí con SQL y resultado reproducible.

In [7]:
SQL = {}

SQL['L1'] = f"""
SELECT :periodo AS periodo, 'L1'::text AS kpi_id,
       COALESCE(SUM({LIC_M_ADJ}),0) AS valor_sql
FROM dw.fact_licitaciones
WHERE {LIC_PERIODO} = :periodo;
"""

SQL['L2'] = f"""
SELECT :periodo AS periodo, 'L2'::text AS kpi_id,
       COUNT(DISTINCT {LIC_BK_COL}) AS valor_sql
FROM dw.fact_licitaciones
WHERE {LIC_PERIODO} = :periodo;
"""

SQL['L3'] = f"""
SELECT :periodo AS periodo, 'L3'::text AS kpi_id,
       COUNT(DISTINCT {LIC_PROV_SK}) AS valor_sql
FROM dw.fact_licitaciones
WHERE {LIC_PERIODO} = :periodo;
"""

SQL['O1'] = f"""
SELECT :periodo AS periodo, 'O1'::text AS kpi_id,
       COALESCE(SUM({OC_MONTO}),0) AS valor_sql
FROM dw.fact_ordenes_compra
WHERE {OC_PERIODO} = :periodo;
"""

SQL['O2'] = f"""
SELECT :periodo AS periodo, 'O2'::text AS kpi_id,
       COUNT(DISTINCT {OC_BK_COL}) AS valor_sql
FROM dw.fact_ordenes_compra
WHERE {OC_PERIODO} = :periodo;
"""

SQL['O3'] = f"""
WITH base AS (
  SELECT {OC_PROV_SK} AS proveedor_sk, SUM({OC_MONTO}) AS monto
  FROM dw.fact_ordenes_compra
  WHERE {OC_PERIODO} = :periodo
  GROUP BY {OC_PROV_SK}
),
tot AS (SELECT COALESCE(SUM(monto),0) AS total FROM base),
top1 AS (SELECT COALESCE(MAX(monto),0) AS top1_monto FROM base)
SELECT :periodo AS periodo, 'O3'::text AS kpi_id,
       CASE WHEN (SELECT total FROM tot)=0 THEN 0
            ELSE (SELECT top1_monto FROM top1) / (SELECT total FROM tot)
       END AS valor_sql;
"""

SQL['C1'] = f"""
SELECT :periodo AS periodo, 'C1_LIC'::text AS kpi_id,
       COALESCE(SUM({LIC_M_ADJ}),0) AS valor_sql
FROM dw.fact_licitaciones
WHERE {LIC_PERIODO} = :periodo
UNION ALL
SELECT :periodo AS periodo, 'C1_OC'::text AS kpi_id,
       COALESCE(SUM({OC_MONTO}),0) AS valor_sql
FROM dw.fact_ordenes_compra
WHERE {OC_PERIODO} = :periodo;
"""

SQL['C2'] = f"""
WITH lic AS (
  SELECT COALESCE(SUM({LIC_M_ADJ}),0) AS lic_monto
  FROM dw.fact_licitaciones
  WHERE {LIC_PERIODO} = :periodo
),
oc AS (
  SELECT COALESCE(SUM({OC_MONTO}),0) AS oc_monto
  FROM dw.fact_ordenes_compra
  WHERE {OC_PERIODO} = :periodo
)
SELECT :periodo AS periodo, 'C2'::text AS kpi_id,
       CASE WHEN (SELECT lic_monto FROM lic)=0 THEN NULL
            ELSE (SELECT oc_monto FROM oc) / (SELECT lic_monto FROM lic)
       END AS valor_sql;
"""

print('SQL templates OK:', list(SQL.keys()))


SQL templates OK: ['L1', 'L2', 'L3', 'O1', 'O2', 'O3', 'C1', 'C2']


## 8) Ejecución batch → Tabla oráculo

Genera:
- `outputs/bi0_oraculo_kpis.csv`
- `outputs/bi0_oraculo_kpis_pivot.csv`
- `outputs/bi0_estado_periodos.csv`

Estos archivos son la base de reconciliación con Power BI.

In [8]:
def run_kpis(periodo: str) -> pd.DataFrame:
    out = []
    for k in ['L1','L2','L3','O1','O2','O3','C2']:
        out.append(read_sql_df(SQL[k], {'periodo': periodo}))
    out.append(read_sql_df(SQL['C1'], {'periodo': periodo}))  # 2 filas
    df_all = pd.concat(out, ignore_index=True)
    est, det = estado_periodo(periodo)
    df_all['estado_periodo'] = est
    df_all['detalle_estado'] = det
    df_all['ts_run'] = datetime.now().isoformat(timespec='seconds')
    return df_all

df_oraculo = pd.concat([run_kpis(p) for p in PERIODOS], ignore_index=True)
display(df_oraculo.head(30))


,periodo,kpi_id,valor_sql,estado_periodo,detalle_estado,ts_run
0,2024-09,L1,4.392265e+12,OK,,2026-01-23T00:19:20
1,2024-09,L2,1.477550e+05,OK,,2026-01-23T00:19:20
2,2024-09,L3,6.684000e+03,OK,,2026-01-23T00:19:20
3,2024-09,O1,1.420518e+12,OK,,2026-01-23T00:19:20
4,2024-09,O2,3.244190e+05,OK,,2026-01-23T00:19:20
5,2024-09,O3,7.309760e-02,OK,,2026-01-23T00:19:20
6,2024-09,C2,3.234134e-01,OK,,2026-01-23T00:19:20
7,2024-09,C1_OC,1.420518e+12,OK,,2026-01-23T00:19:20
8,2024-09,C1_LIC,4.392265e+12,OK,,2026-01-23T00:19:20
9,2024-10,L1,1.115541e+13,OK,,2026-01-23T00:19:22


In [9]:
df_pivot = df_oraculo.pivot_table(
    index=['periodo','estado_periodo','detalle_estado'],
    columns='kpi_id',
    values='valor_sql',
    aggfunc='first'
).reset_index()
display(df_pivot)


kpi_id,periodo,estado_periodo,detalle_estado,C1_LIC,C1_OC,C2,L1,L2,L3,O1,O2,O3
0,2024-09,OK,,4.392265e+12,1.420518e+12,0.323413,4.392265e+12,147755.0,6684.0,1.420518e+12,324419.0,0.073098
1,2024-10,OK,,1.115541e+13,2.458862e+12,0.220419,1.115541e+13,199155.0,9041.0,2.458862e+12,477995.0,0.045844
2,2024-11,OK,,6.411790e+13,1.974168e+12,0.030790,6.411790e+13,475782.0,8072.0,1.974168e+12,429335.0,0.032851
3,2024-12,OK,,3.098688e+13,2.423136e+12,0.078199,3.098688e+13,312983.0,5907.0,2.423136e+12,373131.0,0.066561
4,2025-01,EXCEPCION,LIC_SCHEMA_ANOM,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.000000
5,2025-02,OK,,3.136729e+13,3.629598e+12,0.115713,3.136729e+13,452154.0,6205.0,3.629598e+12,316147.0,0.224015
6,2025-03,NO-OP,OC_SIZE_ANOM,4.489475e+13,0.000000e+00,0.000000,4.489475e+13,485478.0,6813.0,0.000000e+00,0.0,0.000000
7,2025-04,OK,,9.235374e+13,2.487968e+12,0.026940,9.235374e+13,494536.0,6805.0,2.487968e+12,405677.0,0.055980
8,2025-05,OK,,4.230606e+13,2.227616e+12,0.052655,4.230606e+13,470000.0,6602.0,2.227616e+12,376060.0,0.054528
9,2025-06,OK,,6.251789e+13,2.233799e+12,0.035731,6.251789e+13,407938.0,6406.0,2.233799e+12,372131.0,0.055262


In [10]:
out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)

df_oraculo.to_csv(out_dir / 'bi0_oraculo_kpis.csv', index=False)
df_pivot.to_csv(out_dir / 'bi0_oraculo_kpis_pivot.csv', index=False)

df_estado = pd.DataFrame([
    {'periodo': p, 'estado': estado_periodo(p)[0], 'detalle': estado_periodo(p)[1]}
    for p in PERIODOS
])
df_estado.to_csv(out_dir / 'bi0_estado_periodos.csv', index=False)

print('OK -> exports en:', out_dir.resolve())


OK -> exports en: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase5_BI/outputs


## 9) Cierre BI-0

**Listo para BI-1 (exoesqueleto Power BI)** cuando:
- Este notebook corre completo sin errores
- La tabla oráculo se genera en `outputs/`

> Regla: si no cuadra con `bi0_oraculo_kpis*.csv`, el dashboard NO se acepta.